In [0]:
catalog = "venergy"
landing_schema = "landing"
bronze_schema = "bronze"
landing_volume = "landing_volume"
volume_path = f"/Volumes/{catalog}/{landing_schema}/{landing_volume}"

In [0]:
create_catalog = f"CREATE CATALOG IF NOT EXISTS {catalog}"
create_landing_schema = f"CREATE SCHEMA IF NOT EXISTS {catalog}.{landing_schema}"
create_bronze_schema = f"CREATE SCHEMA IF NOT EXISTS {catalog}.{bronze_schema}"
create_landing_volume = f"CREATE VOLUME IF NOT EXISTS {catalog}.{landing_schema}.{landing_volume}"

In [0]:
spark.sql(create_catalog)
spark.sql(create_landing_schema)
spark.sql(create_bronze_schema)
spark.sql(create_landing_volume)

DataFrame[]

In [0]:
from datetime import datetime, timezone
from functools import reduce

from pyspark.sql import functions as F
from pyspark.sql.types import StringType, TimestampType

CATALOG = catalog
BRONZE_SCHEMA = bronze_schema
BRONZE_TABLE = "tb_faturas"

INPUT_PATH = f"{volume_path}/faturas.csv"
TABLE_NAME = f"`{CATALOG}`.`{BRONZE_SCHEMA}`.`{BRONZE_TABLE}`"

spark.conf.set("spark.sql.session.timeZone", "UTC")

print(f"Origem: {INPUT_PATH}")
print(f"Destino: {TABLE_NAME}")

Origem: /Volumes/venergy/landing/landing_volume/faturas.csv
Destino: `venergy`.`bronze`.`tb_faturas`


In [0]:
import csv
import hashlib
from pathlib import Path

SOURCE_COLUMNS = [
    "id_fatura", "id_uc", "mes_referencia", "data_emissao",
    "valor_energia", "tributos", "bandeira_tarifaria", "valor_total",
    "data_vencimento", "data_pagamento", "status",
]


def file_signature(file_path):
    stat = Path(file_path).stat()
    return stat.st_size, stat.st_mtime_ns


def validate_landing_csv(file_path):
    path = Path(file_path)
    if not path.is_file():
        raise ValueError("A origem deve ser um único arquivo CSV existente.")
    signature = file_signature(path)
    record_count = 0
    internal_headers = 0

    try:
        with path.open("r", encoding="utf-8", errors="strict", newline="") as stream:
            reader = csv.reader(
                stream, delimiter=";", quotechar='"', doublequote=True,
                skipinitialspace=False, strict=True,
            )
            if next(reader, None) != SOURCE_COLUMNS:
                raise ValueError("Cabeçalho incompatível: são exigidas as 11 colunas de faturas, na ordem contratada, em UTF-8 sem BOM.")
            for record_count, row in enumerate(reader, start=1):
                if len(row) != len(SOURCE_COLUMNS):
                    raise ValueError(
                        f"Registro {record_count} após o cabeçalho: {len(row)} campos; esperados {len(SOURCE_COLUMNS)}."
                    )
                if any("\x00" in value for value in row):
                    raise ValueError(f"Registro {record_count}: caractere NUL incompatível com o contrato de leitura.")
                internal_headers += row == SOURCE_COLUMNS
    except (csv.Error, UnicodeError) as exc:
        raise ValueError("CSV inválido: verifique a codificação UTF-8 e o fechamento das aspas.") from exc

    if record_count == 0:
        raise ValueError("Extração sem registros: a tabela anterior não deve ser substituída por uma carga vazia.")

    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(chunk)
    if file_signature(path) != signature:
        raise RuntimeError("O arquivo foi alterado durante a pré-validação. Refaça a carga com uma origem estável.")

    return {
        "record_count": record_count,
        "internal_header_count": internal_headers,
        "size_bytes": signature[0],
        "mtime_ns": signature[1],
        "sha256": digest.hexdigest(),
    }

In [0]:
source_profile = validate_landing_csv(INPUT_PATH)
print({key: value for key, value in source_profile.items() if key != "mtime_ns"})

{'record_count': 2184623, 'internal_header_count': 2, 'size_bytes': 206897892, 'sha256': 'dcdb207659b38f7d686fa673151f894dd4f71ff0f4e40f2b70c6208c94b6446e'}


In [0]:
csv_options = {
    "header": "true",
    "sep": ";",
    "encoding": "UTF-8",
    "inferSchema": "false",
    "enforceSchema": "false",
    "quote": '"',
    "escape": '"',
    "multiLine": "true",
    "ignoreLeadingWhiteSpace": "false",
    "ignoreTrailingWhiteSpace": "false",
    "nullValue": "\x00",
    "emptyValue": "",
    "mode": "FAILFAST",
    "unescapedQuoteHandling": "RAISE_ERROR",
    "maxColumns": str(len(SOURCE_COLUMNS)),
}
df_source = spark.read.options(**csv_options).csv(INPUT_PATH)
if df_source.columns != SOURCE_COLUMNS:
    raise ValueError("As colunas lidas pelo Spark não correspondem ao contrato de faturas.")
if not all(isinstance(field.dataType, StringType) for field in df_source.schema.fields):
    raise TypeError("Todas as colunas de origem devem permanecer STRING na Bronze.")

df_source = df_source.na.fill("", subset=SOURCE_COLUMNS)
is_internal_header = reduce(
    lambda left, right: left & right,
    (F.col(column) == F.lit(column) for column in SOURCE_COLUMNS),
)
source_stats = df_source.agg(
    F.count("*").alias("record_count"),
    F.sum(F.when(is_internal_header, 1).otherwise(0)).alias("internal_header_count"),
).first().asDict()
for metric in ("record_count", "internal_header_count"):
    if source_stats[metric] != source_profile[metric]:
        raise RuntimeError(f"Divergência de {metric} entre CSV e Spark; escrita cancelada.")

print(f"Leitura validada: {source_stats}")

Leitura validada: {'record_count': 2184623, 'internal_header_count': 2}


In [0]:
expected_types = [(name, "string") for name in SOURCE_COLUMNS] + [
    ("ingestion_datetime", "timestamp"), ("source_file_path", "string"),
]
if spark.catalog.tableExists(TABLE_NAME):
    detail = spark.sql(f"DESCRIBE DETAIL {TABLE_NAME}").first().asDict()
    if detail["format"].lower() != "delta":
        raise ValueError("O destino existente deve ser uma tabela Delta.")
    actual_types = [
        (field.name, field.dataType.simpleString())
        for field in spark.table(TABLE_NAME).schema.fields
    ]
    if actual_types != expected_types:
        raise ValueError("Schema do destino incompatível; revise a tabela antes de executar a carga.")

expected_signature = (source_profile["size_bytes"], source_profile["mtime_ns"])
if file_signature(INPUT_PATH) != expected_signature:
    raise RuntimeError("A origem mudou desde a validação; escrita cancelada.")

spark.sql(f"CREATE SCHEMA IF NOT EXISTS `{CATALOG}`.`{BRONZE_SCHEMA}`")
ingestion_instant = datetime.now(timezone.utc)
df_bronze = (
    df_source
    .withColumn("ingestion_datetime", F.lit(ingestion_instant).cast(TimestampType()))
    .withColumn("source_file_path", F.lit(INPUT_PATH))
)
(
    df_bronze.write
    .format("delta")
    .mode("overwrite")
    .option("partitionOverwriteMode", "static")
    .saveAsTable(TABLE_NAME)
)

print(f"Gravação concluída; início da ingestão: {ingestion_instant.isoformat()}")

Gravação concluída; início da ingestão: 2026-10-20T23:14:30.776788+00:00


In [0]:
df_written = spark.table(TABLE_NAME)
actual_types = [(field.name, field.dataType.simpleString()) for field in df_written.schema.fields]
if actual_types != expected_types:
    raise RuntimeError("Schema persistido diferente do contrato de faturas Bronze.")

expected_timestamp = F.lit(ingestion_instant).cast(TimestampType())
invalid_metadata = (
    ~F.col("ingestion_datetime").eqNullSafe(expected_timestamp)
    | ~F.col("source_file_path").eqNullSafe(F.lit(INPUT_PATH))
)
written_stats = df_written.agg(
    F.count("*").alias("record_count"),
    F.sum(F.when(is_internal_header, 1).otherwise(0)).alias("internal_header_count"),
    F.sum(F.when(invalid_metadata, 1).otherwise(0)).alias("invalid_metadata_count"),
).first().asDict()
for metric in ("record_count", "internal_header_count"):
    if written_stats[metric] != source_profile[metric]:
        raise RuntimeError(f"Divergência de {metric} na tabela persistida.")
if written_stats["invalid_metadata_count"] != 0:
    raise RuntimeError("Foram persistidas linhas com timestamp ou origem divergente da carga.")
if file_signature(INPUT_PATH) != expected_signature:
    raise RuntimeError("A origem mudou durante a gravação; investigue a versão Delta antes de consumir os dados.")

import json

execution_report = {
    "table": f"{CATALOG}.{BRONZE_SCHEMA}.{BRONZE_TABLE}",
    "source_file_path": INPUT_PATH,
    "source_sha256": source_profile["sha256"],
    "source_size_bytes": source_profile["size_bytes"],
    "record_count": written_stats["record_count"],
    "internal_header_count": written_stats["internal_header_count"],
    "ingestion_datetime_utc": ingestion_instant.isoformat(),
    "write_mode": "overwrite",
    "validation": "passed",
}
print(json.dumps(execution_report, ensure_ascii=False, indent=2))

{
  "table": "venergy.bronze.tb_faturas",
  "source_file_path": "/Volumes/venergy/landing/landing_volume/faturas.csv",
  "source_sha256": "dcdb207659b38f7d686fa673151f894dd4f71ff0f4e40f2b70c6208c94b6446e",
  "source_size_bytes": 206897892,
  "record_count": 2184623,
  "internal_header_count": 2,
  "ingestion_datetime_utc": "2026-10-20T23:14:30.776788+00:00",
  "write_mode": "overwrite",
  "validation": "passed"
}
